# Ejemplo completo: Árbol de Decisión con MLflow

Este notebook toma el ejemplo de clasificación con `bank.csv` y le incorpora MLflow para registrar experimentos, parámetros, métricas, modelo entrenado y artefactos como la imagen del árbol de decisión.

## Objetivo didáctico

Comparar diferentes configuraciones de un árbol de decisión y observar los resultados desde la interfaz de MLflow.

## 2. Importación de librerías

In [38]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

import mlflow
import mlflow.sklearn

## 3. Configuración inicial de MLflow

El experimento se guardará localmente en la carpeta `mlruns/`. Luego podrá visualizarlo ejecutando `mlflow ui` desde la terminal.

In [39]:
mlflow.set_tracking_uri("http://127.0.0.1:9090")
mlflow.set_experiment(experiment_name="arbol_decision_estudiantes")

<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1779324626855, experiment_id='3', last_update_time=1779324626855, lifecycle_stage='active', name='arbol_decision_estudiantes', tags={}, trace_location=None, workspace='default'>

## 4. Carga del dataset


In [40]:
# --- 1. Carga y Preparación del Dataset bank.csv ---
df = pd.read_csv("data/estudiantes.csv")
df.head()

,carrera,modalidad,beca,edad,promedio,asistencias,aprobado
0,Industrial,Presencial,Si,29,5.8,64,Si
1,Industrial,Hibrida,Si,27,6.6,51,No
2,Arquitectura,Presencial,Si,29,8.2,84,Si
3,Economia,Presencial,Si,29,6.6,67,No
4,Economia,Presencial,Si,24,5.1,72,No


## 5. Revisión general del dataset

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   carrera      5000 non-null   object 
 1   modalidad    5000 non-null   object 
 2   beca         5000 non-null   object 
 3   edad         5000 non-null   int64  
 4   promedio     5000 non-null   float64
 5   asistencias  5000 non-null   int64  
 6   aprobado     5000 non-null   object 
dtypes: float64(1), int64(2), object(4)
memory usage: 273.6+ KB


In [42]:
print("Filas y columnas:", df.shape)
df.head()

Filas y columnas: (5000, 7)


,carrera,modalidad,beca,edad,promedio,asistencias,aprobado
0,Industrial,Presencial,Si,29,5.8,64,Si
1,Industrial,Hibrida,Si,27,6.6,51,No
2,Arquitectura,Presencial,Si,29,8.2,84,Si
3,Economia,Presencial,Si,29,6.6,67,No
4,Economia,Presencial,Si,24,5.1,72,No


## 6. Preparación de la variable objetivo

La variable `y` indica si el cliente suscribió o no el depósito. Se transforma:

- `yes` → `1`
- `no` → `0`

In [43]:
df["promedio"] = df["promedio"].map({"Si": 1, "No": 0})

print("--- Distribución de la Variable Objetivo 'y' después del mapeo ---")
print(df["promedio"].value_counts())

--- Distribución de la Variable Objetivo 'y' después del mapeo ---
Series([], Name: count, dtype: int64)


## 7. Codificación de variables categóricas

Se utiliza `pd.get_dummies()` para convertir variables categóricas en variables numéricas.

In [44]:
variables_categorical = [
    "carrera", "modalidad", "beca", "promedio", "aprobado"
]

df_encoded = pd.get_dummies(
    df,
    columns=variables_categorical,
    drop_first=True
)

df_encoded.head()

,edad,asistencias,carrera_Computacion,carrera_Derecho,carrera_Economia,carrera_Industrial,carrera_Medicina,modalidad_Presencial,modalidad_Virtual,beca_Si,aprobado_Si
0,29,64,False,False,False,True,False,True,False,True,True
1,27,51,False,False,False,True,False,False,False,True,False
2,29,84,False,False,False,False,False,True,False,True,True
3,29,67,False,False,True,False,False,True,False,True,False
4,24,72,False,False,True,False,False,True,False,True,False


## 8. Definición de variables predictoras y variable objetivo

Se excluye `duration` porque en muchos análisis de este dataset se considera una variable problemática para predicción previa, ya que se conoce después de realizada la llamada.

In [45]:
print(df_encoded.columns)

Index(['edad', 'asistencias', 'carrera_Computacion', 'carrera_Derecho',
       'carrera_Economia', 'carrera_Industrial', 'carrera_Medicina',
       'modalidad_Presencial', 'modalidad_Virtual', 'beca_Si', 'aprobado_Si'],
      dtype='object')


In [46]:
df_encoded.dtypes

edad                    int64
asistencias             int64
carrera_Computacion      bool
carrera_Derecho          bool
carrera_Economia         bool
carrera_Industrial       bool
carrera_Medicina         bool
modalidad_Presencial     bool
modalidad_Virtual        bool
beca_Si                  bool
aprobado_Si              bool
dtype: object

## 9. División de datos en entrenamiento y prueba

In [47]:

# Revisar columnas
print(df_encoded.columns)

# Crear variables
X = df_encoded.drop(columns=["modalidad"], errors='ignore')
y = df_encoded["carrera_Computacion"]


from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


#X_train, X_test, y_train, y_test = train_test_split(
#    X,
#    y,
#    test_size=0.2,
#    random_state=42,
#    stratify=y
#)

print("--- Tamaños de los Conjuntos de Datos ---")
print(f"Tamaño del conjunto de entrenamiento: {len(X_train)} muestras")
print(f"Tamaño del conjunto de prueba: {len(X_test)} muestras")

Index(['edad', 'asistencias', 'carrera_Computacion', 'carrera_Derecho',
       'carrera_Economia', 'carrera_Industrial', 'carrera_Medicina',
       'modalidad_Presencial', 'modalidad_Virtual', 'beca_Si', 'aprobado_Si'],
      dtype='object')
--- Tamaños de los Conjuntos de Datos ---
Tamaño del conjunto de entrenamiento: 4000 muestras
Tamaño del conjunto de prueba: 1000 muestras


## 10. Experimento individual con MLflow

Esta celda entrena un árbol de decisión y registra en MLflow:

- Parámetros del modelo
- Métricas de evaluación
- Modelo entrenado
- Reporte de clasificación

In [48]:
# Parámetros del árbol de decisión
max_depth = 5
min_samples_leaf = 50
min_samples_split = 100
criterion = "gini"
random_state = 42

with mlflow.start_run(run_name="arbol_decision_estudiantes"):

    decision_tree_model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        criterion=criterion,
        random_state=random_state
    )

    print("--- Entrenando el Modelo de Árbol de Decisión Estudiantes ---")
    decision_tree_model.fit(X_train, y_train)
    print("Entrenamiento completado.")

    y_pred = decision_tree_model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Registrar parámetros en MLflow
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("min_samples_leaf", min_samples_leaf)
    mlflow.log_param("min_samples_split", min_samples_split)
    mlflow.log_param("criterion", criterion)
    mlflow.log_param("random_state", random_state)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("variables_predictoras", X.shape[1])

    # Registrar métricas en MLflow
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    # Guardar el modelo entrenado
    mlflow.sklearn.log_model(
        sk_model=decision_tree_model,
        artifact_path="arbol_decision_estudiantes"
    )

    # Crear y guardar reporte de clasificación como artefacto
    reporte = classification_report(
        y_test,
        y_pred,
        target_names=["No Aprueba (0)", "Aprueba (1)"]
    )

    with open("reporte_clasificacion.txt", "w", encoding="utf-8") as archivo:
        archivo.write(reporte)

    mlflow.log_artifact("reporte_clasificacion.txt")

    print("--- Resultados de la Evaluación del Modelo ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print("Reporte de Clasificación:")
    print(reporte)    

--- Entrenando el Modelo de Árbol de Decisión Estudiantes ---
Entrenamiento completado.


2026/05/20 23:40:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/20 23:40:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


--- Resultados de la Evaluación del Modelo ---
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1-score: 1.0000
Reporte de Clasificación:
                precision    recall  f1-score   support

No Aprueba (0)       1.00      1.00      1.00       839
   Aprueba (1)       1.00      1.00      1.00       161

      accuracy                           1.00      1000
     macro avg       1.00      1.00      1.00      1000
  weighted avg       1.00      1.00      1.00      1000

🏃 View run arbol_decision_estudiantes at: http://127.0.0.1:9090/#/experiments/3/runs/3637fdd071c64603bfefd34682a39ce0
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/3


## 11. Comparación de varios experimentos

Esta parte es útil para clase. Los estudiantes pueden cambiar los hiperparámetros y observar cuál configuración produce mejores resultados.

In [49]:
experimentos = [
    {"max_depth": 3, "min_samples_leaf": 20, "min_samples_split": 50, "criterion": "gini"},
    {"max_depth": 5, "min_samples_leaf": 50, "min_samples_split": 100, "criterion": "gini"},
    {"max_depth": 7, "min_samples_leaf": 30, "min_samples_split": 80, "criterion": "entropy"},
    {"max_depth": 10, "min_samples_leaf": 10, "min_samples_split": 40, "criterion": "gini"},
]

